# Multi-Directional Car License Plate Detection using CNN and YOLOv8

This notebook implements a YOLOv8-based license plate detection workflow designed to detect plates under different angles, rotations, and orientations.

The main goal is to improve detection robustness using:
- YOLOv8 object detection
- Rotation-aware preprocessing
- Bounding box visualization
- IoU-based evaluation

> Note: The dataset is not included in the repository. Download a YOLOv8-compatible license plate dataset and place it inside the `dataset/` folder.

## 1. Install and Import Required Libraries

In [ ]:
# If ultralytics is not installed, uncomment and run the line below
# !pip install ultralytics opencv-python matplotlib numpy pandas

import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from ultralytics import YOLO

print('Libraries imported successfully')

## 2. Dataset Setup

Expected dataset structure:

```text
dataset/
├── data.yaml
├── train/
│   ├── images/
│   └── labels/
├── valid/
│   ├── images/
│   └── labels/
└── test/
    ├── images/
    └── labels/
```

The `data.yaml` file contains dataset paths and class names.

In [ ]:
DATASET_DIR = Path('dataset')
DATA_YAML = DATASET_DIR / 'data.yaml'

print('Dataset directory exists:', DATASET_DIR.exists())
print('data.yaml exists:', DATA_YAML.exists())

if DATA_YAML.exists():
    print('\nDataset YAML path:', DATA_YAML)
else:
    print('\nPlease place your YOLO dataset inside the dataset/ folder.')

## 3. View Sample Training Images

In [ ]:
def show_sample_images(image_dir='dataset/train/images', num_images=4):
    image_dir = Path(image_dir)
    image_files = list(image_dir.glob('*.jpg')) + list(image_dir.glob('*.png')) + list(image_dir.glob('*.jpeg'))

    if not image_files:
        print('No images found in:', image_dir)
        return

    selected_images = image_files[:num_images]

    plt.figure(figsize=(12, 6))
    for i, image_path in enumerate(selected_images):
        image = cv2.imread(str(image_path))
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        plt.subplot(1, len(selected_images), i + 1)
        plt.imshow(image)
        plt.title(image_path.name)
        plt.axis('off')

    plt.tight_layout()
    plt.show()

show_sample_images()

## 4. Rotation-Aware Image Augmentation

License plates may appear tilted due to camera angle, road direction, or vehicle orientation. Rotation-aware augmentation helps the model become more robust to such cases.

In [ ]:
def rotate_image(image, angle):
    height, width = image.shape[:2]
    center = (width // 2, height // 2)

    rotation_matrix = cv2.getRotationMatrix2D(center, angle, 1.0)
    rotated_image = cv2.warpAffine(
        image,
        rotation_matrix,
        (width, height),
        flags=cv2.INTER_LINEAR,
        borderMode=cv2.BORDER_REFLECT
    )

    return rotated_image


def demonstrate_rotation(image_path, angles=(-30, -15, 0, 15, 30)):
    image = cv2.imread(str(image_path))

    if image is None:
        print('Could not read image:', image_path)
        return

    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    plt.figure(figsize=(15, 5))
    for i, angle in enumerate(angles):
        rotated = rotate_image(image_rgb, angle)
        plt.subplot(1, len(angles), i + 1)
        plt.imshow(rotated)
        plt.title(f'Rotation: {angle}°')
        plt.axis('off')

    plt.tight_layout()
    plt.show()


train_images = list(Path('dataset/train/images').glob('*.jpg')) + list(Path('dataset/train/images').glob('*.png'))

if train_images:
    demonstrate_rotation(train_images[0])
else:
    print('Add dataset images to dataset/train/images to test rotation augmentation.')

## 5. Load YOLOv8 Model

In [ ]:
# yolov8n.pt is lightweight and suitable for quick experimentation.
# For better accuracy, try yolov8s.pt or yolov8m.pt.
model = YOLO('yolov8n.pt')
print('YOLOv8 model loaded successfully')

## 6. Train YOLOv8 Model

The dataset must be in YOLO format with a valid `data.yaml` file.

For quick testing, use fewer epochs. For better performance, increase `epochs`.

In [ ]:
if DATA_YAML.exists():
    results = model.train(
        data=str(DATA_YAML),
        epochs=10,
        imgsz=640,
        batch=8,
        name='license_plate_yolov8'
    )
else:
    print('Training skipped because dataset/data.yaml was not found.')

## 7. Run Inference on Test Images

After training, use the best trained model from `runs/detect/license_plate_yolov8/weights/best.pt`. If training is not performed, the default YOLOv8 model is used for demonstration.

In [ ]:
best_model_path = Path('runs/detect/license_plate_yolov8/weights/best.pt')

if best_model_path.exists():
    detection_model = YOLO(str(best_model_path))
    print('Loaded trained model:', best_model_path)
else:
    detection_model = model
    print('Trained model not found. Using base YOLOv8 model for demonstration.')

In [ ]:
test_images = list(Path('dataset/test/images').glob('*.jpg')) + list(Path('dataset/test/images').glob('*.png')) + list(Path('dataset/test/images').glob('*.jpeg'))

if test_images:
    sample_test_image = test_images[0]
    inference_results = detection_model.predict(source=str(sample_test_image), conf=0.25)
    print('Inference completed on:', sample_test_image)
else:
    inference_results = []
    print('No test images found in dataset/test/images.')

## 8. Bounding Box Visualization

In [ ]:
def visualize_predictions(results):
    if not results:
        print('No prediction results available.')
        return

    for result in results:
        annotated_image = result.plot()
        annotated_image = cv2.cvtColor(annotated_image, cv2.COLOR_BGR2RGB)

        plt.figure(figsize=(10, 8))
        plt.imshow(annotated_image)
        plt.title('License Plate Detection Result')
        plt.axis('off')
        plt.show()

visualize_predictions(inference_results)

## 9. IoU Evaluation Helper

Intersection over Union (IoU) measures overlap between predicted and ground-truth bounding boxes.

In [ ]:
def calculate_iou(box_a, box_b):
    x_a = max(box_a[0], box_b[0])
    y_a = max(box_a[1], box_b[1])
    x_b = min(box_a[2], box_b[2])
    y_b = min(box_a[3], box_b[3])

    intersection = max(0, x_b - x_a) * max(0, y_b - y_a)

    area_a = max(0, box_a[2] - box_a[0]) * max(0, box_a[3] - box_a[1])
    area_b = max(0, box_b[2] - box_b[0]) * max(0, box_b[3] - box_b[1])

    union = area_a + area_b - intersection

    if union == 0:
        return 0.0

    return intersection / union


example_pred_box = [50, 50, 200, 120]
example_true_box = [60, 55, 210, 130]

iou_score = calculate_iou(example_pred_box, example_true_box)
print('Example IoU:', round(iou_score, 4))

## 10. Evaluate Model on Validation Set

In [ ]:
if DATA_YAML.exists():
    metrics = detection_model.val(data=str(DATA_YAML))
    print(metrics)
else:
    print('Validation skipped because dataset/data.yaml was not found.')

## 11. Save Sample Output

This cell saves a prediction visualization in the `sample_outputs/` folder for README/demo use.

In [ ]:
output_dir = Path('sample_outputs')
output_dir.mkdir(exist_ok=True)

if inference_results:
    result = inference_results[0]
    annotated_image = result.plot()
    output_path = output_dir / 'sample_detection_result.jpg'
    cv2.imwrite(str(output_path), annotated_image)
    print('Saved sample output to:', output_path)
else:
    print('No inference result available to save.')

## 12. Conclusion

This project demonstrates a YOLOv8-based workflow for multi-directional license plate detection. The notebook includes dataset setup, rotation-aware augmentation, model training, inference, visualization, and IoU evaluation.

### Future Improvements
- Add OCR for reading detected plates
- Use rotated bounding boxes for stronger angle awareness
- Deploy the trained model using Streamlit or FastAPI
- Add real-time video inference
- Train with larger multi-angle datasets